# WildfireSpreadTS v3

Two experiments, 3-fold CV, bs=128, 20 epochs each:
- **Exp 1 — UTAE_baseline**: exact paper implementation (`cfgs/UTAE/all_features.yaml`)
- **Exp 2 — UTAE_pretrained_CNN**: two-stage training — pretrain spatial CNN encoder on T=1, then freeze it and train temporal attention on T=5

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import os

# Clone WildfireSpreadTS repo (provides dataloader + UTAELightning)
if not os.path.exists('/content/WildfireSpreadTS'):
    !git clone --quiet https://github.com/SebastianGer/WildfireSpreadTS.git /content/WildfireSpreadTS

!pip install -q segmentation-models-pytorch pytorch-lightning==2.0.1 \
    torchmetrics einops jsonargparse[signatures]==4.20.1 wandb h5py rasterio

import sys
sys.path.insert(0, '/content/WildfireSpreadTS')
sys.path.insert(0, '/content/WildfireSpreadTS/src')

os.environ['WANDB_MODE'] = 'disabled'
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

# Fix removed T_co import in newer torch versions
!sed -i 's/from torch.utils.data.dataset import T_co/T_co = None/' \
    /content/WildfireSpreadTS/src/dataloader/FireSpreadDataset.py

print('Setup complete.')

Setup complete.


In [2]:
import json
import warnings

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint

warnings.filterwarnings('ignore')
torch.set_float32_matmul_precision('medium')

from dataloader.FireSpreadDataModule import FireSpreadDataModule
from models.UTAELightning import UTAELightning

In [3]:
import shutil, time
import os, glob
DATA_DIR = "/content/drive/MyDrive/Climate Change - Project/WildfireSpreadTS_10pct"   # <── edit if needed
RESULTS_DIR    = "/content/drive/MyDrive/Climate Change - Project/models/v3"


# Copy HDF5 files from Drive → local SSD (eliminates Drive FUSE bottleneck)
LOCAL_DATA = '/content/wildfire_local'
os.makedirs(LOCAL_DATA, exist_ok=True)

t0 = time.time()
for year_link in glob.glob(os.path.join(DATA_DIR, '*')):
    year = os.path.basename(year_link)
    dst_year = os.path.join(LOCAL_DATA, year)
    os.makedirs(dst_year, exist_ok=True)
    for src in glob.glob(os.path.join(year_link, '*.hdf5')):
        dst = os.path.join(dst_year, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    n = len(glob.glob(os.path.join(dst_year, '*.hdf5')))
    print(f"  {year}: {n} files copied")

print(f"Done in {time.time()-t0:.0f}s")

# Switch DATA_DIR to local SSD copy
DATA_DIR = LOCAL_DATA
print(f"DATA_DIR → {DATA_DIR}")


  train: 0 files copied
  val: 0 files copied
  test: 0 files copied
  split_info.json: 0 files copied
  subset_manifest.json: 0 files copied
  3fold_A_results.json: 0 files copied
  3fold_B_results.json: 0 files copied
  3fold_C_results.json: 0 files copied
Done in 0s
DATA_DIR → /content/wildfire_local


In [4]:
import os, glob, shutil, time

# Step 1 — symlink all year folders into one flat dir (to find files across train/val/test)
_base = glob.glob('/content/drive/MyDrive/Climate Change - Project/WildfireSpreadTS_10pct')[0]
_merged = '/content/wildfire_merged'
os.makedirs(_merged, exist_ok=True)

for split_dir in glob.glob(os.path.join(_base, '*')):
    for year_dir in glob.glob(os.path.join(split_dir, '*')):
        if not os.path.basename(year_dir).isdigit():
            continue   # skip split_info.json etc.
        link = os.path.join(_merged, os.path.basename(year_dir))
        if not os.path.exists(link):
            os.symlink(year_dir, link)

# Step 2 — copy from symlinks → local SSD (eliminates Drive FUSE bottleneck)
DATA_DIR = '/content/wildfire_local'
os.makedirs(DATA_DIR, exist_ok=True)
t0 = time.time()
for year_link in sorted(glob.glob(os.path.join(_merged, '*'))):
    year = os.path.basename(year_link)
    dst_year = os.path.join(DATA_DIR, year)
    os.makedirs(dst_year, exist_ok=True)
    for src in glob.glob(os.path.join(year_link, '*.hdf5')):
        dst = os.path.join(dst_year, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    n = len(glob.glob(os.path.join(dst_year, '*.hdf5')))
    print(f"  {year}: {n} fires on local SSD")
print(f"Copy done in {time.time()-t0:.0f}s")
print(f"DATA_DIR → {DATA_DIR}")

RESULTS_DIR = '/content/drive/MyDrive/Climate Change - Project/models/v3'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Training budget ─────────────────────────────────────────────────────────────
BATCH_SIZE  = 128
MAX_EPOCHS  = 20
NUM_WORKERS = 8
N_DAYS      = 5
CROP_SIZE   = 128
N_CHANNELS  = 40

# ── Paper hyperparams (cfgs/UTAE/all_features.yaml) ────────────────────────────
LR               = 0.01
POS_CLASS_WEIGHT = 10

# ── 3-fold CV ──────────────────────────────────────────────────────────────────
FOLD_IDS   = [0, 3, 10]
FOLD_NAMES = [
    'A  train[18,19] / val[20] / test[21]',
    'B  train[18,20] / val[21] / test[19]',
    'C  train[20,21] / val[18] / test[19]',
]


  2018: 17 fires on local SSD
  2019: 7 fires on local SSD
  2020: 20 fires on local SSD
  2021: 15 fires on local SSD
Copy done in 0s
DATA_DIR → /content/wildfire_local


In [5]:
def get_datamodule(fold_id, batch_size=4, n_days=N_DAYS):
    return FireSpreadDataModule(
        data_dir=DATA_DIR,
        n_leading_observations=n_days,
        crop_side_length=128,
        load_from_hdf5=True,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        data_fold_id=fold_id,
        remove_duplicate_features=False,
        features_to_keep=None,
    )

print('get_datamodule defined.')


def get_alldata_datamodule(batch_size=4):
    """
    Datamodule that treats ALL years as training data.
    We reuse fold 0 but override val/test splits to be empty,
    achieved by using a dedicated fold where train = all 4 years.

    Practical approach: use fold_id=0 for its dataloader structure,
    then concatenate train+val+test into one dataset for AE training.
    We do this by subclassing and overriding setup().
    """
    class AllDataModule(FireSpreadDataModule):
        def setup(self, stage=None):
            # Call parent to build the standard splits
            super().setup(stage)
            # Concatenate all splits into train — AE sees everything
            from torch.utils.data import ConcatDataset
            all_ds = ConcatDataset([
                self.train_dataset,
                self.val_dataset,
                self.test_dataset,
            ])
            self.train_dataset = all_ds
            # Val uses just original val (small, for reconstruction loss monitoring)
            # test_dataset left as-is (unused during AE training)

    dm = AllDataModule(
        data_dir=DATA_DIR,
        n_leading_observations=5,
        crop_side_length=128,
        load_from_hdf5=True,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        data_fold_id=0,          # fold 0 just to initialise — then overridden above
        remove_duplicate_features=False,
        features_to_keep=None,
    )
    return dm


def get_pos_class_weight(fold_id):
    """
    Positive class weight for BCE loss — matches UTAELightning's paper config.
    Computed as inverse fire pixel frequency in the training set.
    Using 500 as a safe default matching the paper's weighted BCE setup.
    """
    return 500.0


print('Datamodule helpers defined.')


get_datamodule defined.
Datamodule helpers defined.


## Experiment 1 — UTAE_baseline
Exact paper config: `loss=BCE`, `pos_class_weight=10`, `lr=0.01`, `AdamW`, `32-bit`.

In [6]:
from dataloader.utils import get_means_stds_missing_values

def get_pos_class_weight(fold_id):
    """Replicate train.py before_instantiate_classes: compute pos_class_weight from training fire rate."""
    train_years, _, _ = FireSpreadDataModule.split_fires(fold_id)
    _, _, missing_values_rates = get_means_stds_missing_values(train_years)
    fire_rate = 1 - missing_values_rates[-1]   # last channel = active fire
    w = float(1 / fire_rate)
    print(f"  Fold {fold_id} train_years={train_years}  fire_rate={fire_rate:.5f}  pos_class_weight={w:.1f}")
    return w


class UTAEWrapper(UTAELightning):
    """
    Exact paper UTAE: pos_class_weight computed per-fold from fire rate (matches train.py).
    Only adds configure_optimizers + on_test_epoch_end (removes wandb).
    """
    def __init__(self, n_channels=N_CHANNELS, pos_class_weight=500.0, lr=LR):
        super().__init__(
            n_channels=n_channels,
            flatten_temporal_dimension=False,
            pos_class_weight=pos_class_weight,
            loss_function='BCE',
                use_doy=False   # ← ADD
        )
        self._lr = lr

    def configure_optimizers(self):
        return {'optimizer': torch.optim.AdamW(
            self.parameters(), lr=self._lr, betas=(0.9, 0.999), weight_decay=0.01
        )}

    def on_test_epoch_end(self):
        ap = self.test_avg_precision.compute()
        self.log('test_ap', ap)
        print(f'  Test AP: {ap.item():.4f}')


In [7]:
# def run_utae_fold(fold_id, fold_name):
#     print(f"\n{'='*60}")
#     print(f'UTAE_baseline | Fold {fold_id}: {fold_name}')
#     print(f"{'='*60}")

#     pcw = get_pos_class_weight(fold_id)   # ~960, not 10
#     model    = UTAEWrapper(pos_class_weight=pcw)
#     dm       = get_datamodule(fold_id)
#     ckpt_dir = os.path.join(RESULTS_DIR, 'UTAE_baseline', f'fold_{fold_id}')
#     trainer, ckpt_cb = make_trainer(ckpt_dir, MAX_EPOCHS)

#     trainer.fit(model, datamodule=dm)
#     results = trainer.test(model, datamodule=dm, ckpt_path='best')
#     ap = float(results[0].get('test_ap', results[0].get('test_AP', 0.0)))

#     json.dump({'fold_id': fold_id, 'test_ap': ap, 'pos_class_weight': pcw},
#               open(os.path.join(ckpt_dir, 'result.json'), 'w'), indent=2)
#     print(f'  → AP = {ap:.4f}')
#     return ap


# utae_results = {}
# for fold_id, fold_name in zip(FOLD_IDS, FOLD_NAMES):
#     utae_results[fold_id] = run_utae_fold(fold_id, fold_name)

# aps = list(utae_results.values())
# print(f"\nUTAE_baseline  mean AP = {sum(aps)/len(aps):.4f}")
# print(f"(paper 12-fold SOTA: 0.372 ± 0.088)")


## Experiment 2 — UTAE_pretrained_CNN

Two-stage training (total budget = 20 epochs, same as Exp 1):

| Stage | Epochs | Input T | What trains |
|-------|--------|---------|-------------|
| 1 | 10 | T=1 (last frame) | Spatial CNN encoder (`in_conv` + `down_blocks`) + decoder — no temporal signal |
| 2 | 10 | T=5 | Temporal encoder (LTAE) + decoder — spatial CNN frozen |

Rationale: Stage 1 teaches the per-frame spatial features to recognise fire without leaning on temporal context. Stage 2 then learns *when* across time given fixed spatial representations.

In [8]:
class UTAEPretrainedCNN(UTAELightning):
    """
    UTAE with two-stage training:
      Stage 1 — train on T=1 to pretrain the spatial CNN encoder
      Stage 2 — freeze spatial encoder (in_conv + down_blocks), train temporal attention on T=5
    """
    def __init__(self, n_channels=N_CHANNELS, pos_class_weight=POS_CLASS_WEIGHT, lr=LR):
        super().__init__(
            n_channels=n_channels,
            flatten_temporal_dimension=False,
            pos_class_weight=pos_class_weight,
            loss_function='BCE',
            # use_doy omitted — UTAELightning already passes use_doy=True to BaseModel
        )
        self._lr = lr

    def configure_optimizers(self):
        # Only optimise parameters that are still trainable (respects freeze_spatial_encoder)
        trainable = [p for p in self.parameters() if p.requires_grad]
        return {'optimizer': torch.optim.AdamW(
            trainable, lr=self._lr, betas=(0.9, 0.999), weight_decay=0.01
        )}

    def freeze_spatial_encoder(self):
        """Freeze in_conv and down_blocks — the per-frame spatial CNN encoder."""
        for param in self.model.in_conv.parameters():
            param.requires_grad = False
        for param in self.model.down_blocks.parameters():
            param.requires_grad = False
        frozen    = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  Frozen params: {frozen:,}  |  Trainable params: {trainable:,}')

    def on_test_epoch_end(self):
        ap = self.test_avg_precision.compute()
        self.log('test_ap', ap)
        print(f'  Test AP: {ap.item():.4f}')


In [9]:
# STAGE1_EPOCHS = 10
# STAGE2_EPOCHS = 10


# def run_pretrained_utae_fold(fold_id, fold_name):
#     print(f"\n{'='*60}")
#     print(f'UTAE_pretrained_CNN | Fold {fold_id}: {fold_name}')
#     print(f"{'='*60}")

#     pcw = get_pos_class_weight(fold_id)

#     # ── Stage 1: pretrain spatial CNN (T=1) ────────────────────────────────────
#     print(f'\n  [Stage 1] Pretrain spatial CNN — {STAGE1_EPOCHS} epochs, T=1')
#     model1 = UTAEPretrainedCNN(pos_class_weight=pcw)
#     dm1    = get_datamodule(fold_id, n_days=1)
#     s1_dir = os.path.join(RESULTS_DIR, 'UTAE_pretrained_CNN', f'fold_{fold_id}', 'stage1')
#     trainer1, ckpt_cb1 = make_trainer(s1_dir, STAGE1_EPOCHS)
#     trainer1.fit(model1, datamodule=dm1)

#     # ── Stage 2: freeze CNN, train temporal attention (T=5) ────────────────────
#     print(f'\n  [Stage 2] Freeze CNN, train temporal attention — {STAGE2_EPOCHS} epochs, T=5')
#     model2 = UTAEPretrainedCNN.load_from_checkpoint(
#         ckpt_cb1.best_model_path,
#         n_channels=N_CHANNELS, pos_class_weight=pcw, lr=LR,
#     )
#     model2.freeze_spatial_encoder()

#     dm2    = get_datamodule(fold_id, n_days=N_DAYS)
#     s2_dir = os.path.join(RESULTS_DIR, 'UTAE_pretrained_CNN', f'fold_{fold_id}', 'stage2')
#     trainer2, ckpt_cb2 = make_trainer(s2_dir, STAGE2_EPOCHS)
#     trainer2.fit(model2, datamodule=dm2)
#     results = trainer2.test(model2, datamodule=dm2, ckpt_path='best')
#     ap = float(results[0].get('test_ap', results[0].get('test_AP', 0.0)))

#     json.dump({'fold_id': fold_id, 'test_ap': ap, 'pos_class_weight': pcw},
#               open(os.path.join(s2_dir, 'result.json'), 'w'), indent=2)
#     print(f'  → AP = {ap:.4f}')
#     return ap


# pretrained_results = {}
# for fold_id, fold_name in zip(FOLD_IDS, FOLD_NAMES):
#     pretrained_results[fold_id] = run_pretrained_utae_fold(fold_id, fold_name)

# aps = list(pretrained_results.values())
# print(f"\nUTAE_pretrained_CNN  mean AP = {sum(aps)/len(aps):.4f}")


In [32]:
# Run this first to find the correct path
import subprocess
result = subprocess.run(
    ['find', '/content/WildfireSpreadTS', '-name', 'utae.py'],
    capture_output=True, text=True
)
print(result.stdout)

import torch.nn as nn

/content/WildfireSpreadTS/src/models/utae_paps_models/utae.py



In [33]:
# First add the __init__.py so Python treats it as a package
import os
open('/content/WildfireSpreadTS/src/models/utae_paps_models/__init__.py', 'a').close()
from models.utae_paps_models.utae import ConvBlock, DownConvBlock

class ConvAE(pl.LightningModule):
    """
    Autoencoder whose encoder (in_conv + down_blocks) is architecturally
    identical to UTAE's spatial encoder. Trained with MSE reconstruction
    on all T input frames — no fire labels used.
    Only the encoder is transferred to UTAE after training.
    """
    def __init__(self, n_channels=N_CHANNELS, lr=LR):
        super().__init__()
        self.save_hyperparameters()

        self.in_conv = ConvBlock(
            nkernels=[n_channels, 64, 64],
            pad_value=0, norm='group', padding_mode='reflect',
        )
        self.down_blocks = nn.ModuleList([
            DownConvBlock(d_in=64,  d_out=64,  k=4, s=2, p=1,
                          pad_value=0, norm='group', padding_mode='reflect'),
            DownConvBlock(d_in=64,  d_out=64,  k=4, s=2, p=1,
                          pad_value=0, norm='group', padding_mode='reflect'),
            DownConvBlock(d_in=64,  d_out=128, k=4, s=2, p=1,
                          pad_value=0, norm='group', padding_mode='reflect'),
        ])
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.GroupNorm(4, 64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 64, 4, stride=2, padding=1),
            nn.GroupNorm(4, 64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 64, 4, stride=2, padding=1),
            nn.GroupNorm(4, 64), nn.ReLU(inplace=True),
            nn.Conv2d(64, n_channels, kernel_size=1),
        )

    def encode(self, x):
        z = self.in_conv(x)
        for d in self.down_blocks:
            z = d(z)
        return z

    def forward(self, x):
        return self.decoder(self.encode(x))

    def _mse_step(self, batch):
        x = batch[0]                    # (B, T, C, H, W)
        B, T, C, H, W = x.shape
        frames = x.reshape(B * T, C, H, W)   # train on ALL frames, not just last
        return F.mse_loss(self(frames), frames)

    def training_step(self, batch, _):
        loss = self._mse_step(batch)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        loss = self._mse_step(batch)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


In [12]:
import inspect
print(inspect.getsource(UTAEWithAEEncoder.__init__))

NameError: name 'UTAEWithAEEncoder' is not defined

In [13]:
dm = get_datamodule(0)
dm.setup("fit")
batch = next(iter(dm.train_dataloader()))
print(len(batch))
for i, b in enumerate(batch):
    print(f"  {i}: {b.shape}")

TypeError: FireSpreadDataModule.__init__() missing 1 required positional argument: 'n_leading_observations_test_adjustment'

In [14]:
!sed -i 's/use_doy=True/use_doy=False/' \
    /content/WildfireSpreadTS/src/models/UTAELightning.py

In [34]:
class UTAEWithAEEncoder(UTAELightning):
    """
    UTAE with its spatial encoder (in_conv + down_blocks) replaced by
    weights from a pre-trained ConvAE and frozen.
    L-TAE + decoder remain fully trainable.
    Everything else is identical to UTAEWrapper (same loss, same optimiser).
    """
    def __init__(self, n_channels=N_CHANNELS, pos_class_weight=500.0, lr=LR):
        super().__init__(
            n_channels=n_channels,
            flatten_temporal_dimension=False,
            pos_class_weight=pos_class_weight,
            loss_function='BCE'
        )
        self._lr = lr

    def load_and_freeze_ae_encoder(self, ae_ckpt_path: str):
        ae = ConvAE.load_from_checkpoint(ae_ckpt_path, n_channels=N_CHANNELS)
        self.model.in_conv.load_state_dict(ae.in_conv.state_dict())
        self.model.down_blocks.load_state_dict(ae.down_blocks.state_dict())
        del ae
        for p in self.model.in_conv.parameters():
            p.requires_grad = False
        for p in self.model.down_blocks.parameters():
            p.requires_grad = False
        frozen    = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  Encoder frozen: {frozen:,}  |  Trainable (L-TAE + decoder): {trainable:,}')

    def configure_optimizers(self):
        trainable = [p for p in self.parameters() if p.requires_grad]
        return {'optimizer': torch.optim.AdamW(
            trainable, lr=self._lr, betas=(0.9, 0.999), weight_decay=0.01
        )}

    def on_test_epoch_end(self):
        ap = self.test_avg_precision.compute()
        self.log('test_ap', ap)
        print(f'  Test AP (UTAE_AE_encoder): {ap.item():.4f}')

    def forward(self, x: torch.Tensor, doys=None) -> torch.Tensor:
        B, T, C, H, W = x.shape
        dummy_doys = torch.zeros(B, T, dtype=x.dtype, device=x.device)
        return self.model(x, batch_positions=dummy_doys, return_att=False)

In [35]:

AE_EPOCHS   = 50   # stage 1: ConvAE reconstruction (no labels)
UTAE_EPOCHS = 20   # stage 2: fire prediction with frozen AE encoder
# Total = 40 epochs — same budget as UTAE_baseline

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import torch.nn.functional as F
def make_trainer(out_dir, max_epochs):
    os.makedirs(out_dir, exist_ok=True)
    ckpt_cb = ModelCheckpoint(
        dirpath=out_dir,
        filename='best',
        monitor='val_loss',
        mode='min',
        save_top_k=1,
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=10, mode='min')
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=[ckpt_cb, early_stop],
        accelerator='auto',
        devices=1,
        precision='16-mixed',
        log_every_n_steps=5,
        enable_progress_bar=True,
    )
    return trainer, ckpt_cb

In [36]:
from torch.utils.data import ConcatDataset

def get_datamodule(fold_id, batch_size=4, n_days=N_DAYS):
    return FireSpreadDataModule(
        data_dir=DATA_DIR,
        n_leading_observations=n_days,
        n_leading_observations_test_adjustment=N_DAYS,
        crop_side_length=128,
        load_from_hdf5=True,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
        data_fold_id=fold_id,
        remove_duplicate_features=False,
        features_to_keep=None,
    )

# ── Stage 1: pre-train ConvAE once on ALL years ──────────────────────────────
AE_CKPT_DIR  = os.path.join(RESULTS_DIR, 'UTAE_AE_encoder', 'ae_pretrain')
AE_BEST_CKPT = os.path.join(AE_CKPT_DIR, 'best.ckpt')

if os.path.exists(AE_BEST_CKPT):
    print(f'AE checkpoint found — skipping pre-training.\n  {AE_BEST_CKPT}')
else:
    print('=' * 60)
    print('Stage 1: ConvAE pre-training on ALL years (2018–2021)')
    print('=' * 60)

    # Use fold 0 directly — train=2018+2019, val=2020, test=2021
    # covers all 4 years with correct cropping already applied by the datamodule
    _dm0 = get_datamodule(0, batch_size=BATCH_SIZE)

    ae_model   = ConvAE()
    ae_trainer, ae_ckpt_cb = make_trainer(AE_CKPT_DIR, AE_EPOCHS)
    ae_trainer.fit(ae_model, datamodule=_dm0)

    # The ModelCheckpoint callback already saves the best model to AE_BEST_CKPT
    # shutil.copy2(ae_ckpt_cb.best_model_path, AE_BEST_CKPT) # This line caused SameFileError
    print(f'\nAE pre-training complete → {AE_BEST_CKPT}')
    del ae_model, _dm0


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name        | Type       | Params
-------------------------------------------
0 | in_conv     | ConvBlock  | 60.3 K
1 | down_blocks | ModuleList | 567 K 
2 | decoder     | Sequential | 265 K 
-------------------------------------------
892 K     Trainable params
0         Non-trainable params
892 K     Total params
3.572     Total estimated model params size (MB)


Stage 1: ConvAE pre-training on ALL years (2018–2021)
Using the following dataset split:
Train years: [2018, 2019], Val years: [2020], Test years: [2021]


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]


AE pre-training complete → /content/drive/MyDrive/Climate Change - Project/models/v3/UTAE_AE_encoder/ae_pretrain/best.ckpt


In [37]:


# ── Stage 2: fine-tune UTAE with frozen AE encoder across 3 folds ────────────
ae_utae_results = {}
for fold_id, fold_name in zip(FOLD_IDS, FOLD_NAMES):
    print(f"\n{'='*60}")
    print(f'UTAE_AE_encoder | Fold {fold_id}: {fold_name}')
    print(f"{'='*60}")

    pcw   = get_pos_class_weight(fold_id)
    dm    = get_datamodule(fold_id)

    model = UTAEWithAEEncoder(pos_class_weight=pcw)
    model.load_and_freeze_ae_encoder(AE_BEST_CKPT)

    utae_dir = os.path.join(RESULTS_DIR, 'UTAE_AE_encoder', f'fold_{fold_id}', 'stage2_utae')
    utae_trainer, utae_ckpt_cb = make_trainer(utae_dir, UTAE_EPOCHS)
    utae_trainer.fit(model, datamodule=dm)
    results = utae_trainer.test(model, datamodule=dm, ckpt_path='best')
    ap = float(results[0].get('test_ap', results[0].get('test_AP', 0.0)))

    json.dump({'fold_id': fold_id, 'test_ap': ap, 'pos_class_weight': pcw},
              open(os.path.join(utae_dir, 'result.json'), 'w'), indent=2)
    print(f'  → AP = {ap:.4f}')
    ae_utae_results[fold_id] = ap

aps = list(ae_utae_results.values())
print(f"\nUTAE_AE_encoder  mean AP = {sum(aps)/len(aps):.4f}")

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name               | Type                       | Params
-------------------------------------------------------------------
0  | loss               | BCEWithLogitsLoss          | 0     
1  | train_f1           | BinaryF1Score              | 0     
2  | val_f1             | BinaryF1Score              | 0     
3  | test_f1            | BinaryF1Score              | 0     
4  | test_avg_precision | BinaryAveragePrecision     | 0     
5  | test_precisio


UTAE_AE_encoder | Fold 0: A  train[18,19] / val[20] / test[21]
Using the following dataset split:
Train years: [2018, 2019], Val years: [2020], Test years: [2021]
  Fold 0 train_years=[2018, 2019]  fire_rate=0.00104  pos_class_weight=964.3
  Encoder frozen: 627,648  |  Trainable (L-TAE + decoder): 471,363
Using the following dataset split:
Train years: [2018, 2019], Val years: [2020], Test years: [2021]


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Using the following dataset split:
Train years: [2020, 2021], Val years: [2018], Test years: [2019]


Testing: 0it [00:00, ?it/s]

  Test AP (UTAE_AE_encoder): 0.0034


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Runningstage.testing    ┃                           ┃
┃          metric           ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_AP          │   0.0034351409412920475   │
│          test_ap          │   0.0034351409412920475   │
│          test_f1          │   0.0010268460027873516   │
│         test_iou          │   0.0005136867403052747   │
│         test_loss         │     1.033984661102295     │
│      test_precision       │   0.0005139964050613344   │
│        test_recall        │    0.4602135717868805     │
└───────────────────────────┴───────────────────────────┘

  → AP = 0.0034

UTAE_AE_encoder  mean AP = 0.0319


In [ ]:
rows = [
    ('UTAE_baseline',    utae_results),
]

print(f"{'Model':<25} {'Fold A':>8} {'Fold B':>8} {'Fold C':>8} {'Mean AP':>9}")
print('-' * 62)
for name, res in rows:
    vals = [res[fid] for fid in FOLD_IDS]
    print(f"{name:<25} {vals[0]:>8.4f} {vals[1]:>8.4f} {vals[2]:>8.4f} {sum(vals)/3:>9.4f}")
print('-' * 62)
print(f"{'Paper UTAE (12-fold)':<25} {'':>8} {'':>8} {'':>8} {'0.3720':>9}")


## Results Summary

In [38]:
rows = [
    ('UTAE_baseline',       utae_results),
    ('UTAE_pretrained_CNN', pretrained_results),
    ('UTAE_AE_encoder',  ae_utae_results)
]

print(f"{'Model':<25} {'Fold A':>8} {'Fold B':>8} {'Fold C':>8} {'Mean AP':>9}")
print('-' * 62)
for name, res in rows:
    vals = [res[fid] for fid in FOLD_IDS]
    print(f"{name:<25} {vals[0]:>8.4f} {vals[1]:>8.4f} {vals[2]:>8.4f} {sum(vals)/3:>9.4f}")
print('-' * 62)
print(f"{'Paper UTAE (12-fold)':<25} {'':>8} {'':>8} {'':>8} {'0.3720':>9}")

NameError: name 'utae_results' is not defined